# Olist E-commerce Analysis · Cleaned Notebook

이 노트북은 복구된 원본 분석 노트북을 포트폴리오 검토용으로 정리한 버전입니다.

- 실행 출력과 환경 의존 메타데이터를 제거했습니다.
- 원본에 있던 데이터 로딩, 테이블 병합, 결측 처리, 배송 지연 파생변수, 지역/카테고리 EDA 흐름을 유지했습니다.
- 원본의 단순 payment merge는 주문 중복을 만들 수 있어, 실제 재현 가능한 전처리는 `src/preprocessing.py`의 order-level payment aggregation을 권장합니다.
- 원본 CSV는 저장소에 포함하지 않습니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data")

## 1. Load source tables

복구된 원본 노트북에서 사용한 Olist CSV 이름을 그대로 사용합니다.

In [ ]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
geo = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv", encoding="ISO-8859-1")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

## 2. Column selection and missing values

원본 노트북에서 분석 목적상 제외한 컬럼과 결측 처리 로직입니다.

In [ ]:
customers.drop(columns=[c for c in ["customer_zip_code_prefix"] if c in customers.columns], inplace=True)
geo.drop(columns=[c for c in ["geolocation_zip_code_prefix"] if c in geo.columns], inplace=True)
payments.drop(
    columns=[c for c in ["payment_sequential", "payment_type", "payment_installments"] if c in payments.columns],
    inplace=True,
)
reviews.drop(
    columns=[c for c in ["review_id", "review_comment_title", "review_creation_date"] if c in reviews.columns],
    inplace=True,
)
orders.drop(columns=[c for c in ["order_approved_at"] if c in orders.columns], inplace=True)
products.drop(
    columns=[
        c for c in [
            "product_name_lenght",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ] if c in products.columns
    ],
    inplace=True,
)

products = products.dropna()
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")

## 3. Merge tables

아래 셀은 복구된 원본 notebook의 병합 흐름을 보존합니다.

> 주의: payment가 주문당 여러 행이면 order/item과 결합할 때 주문이 증식할 수 있습니다. 정리된 파이프라인은 `src/preprocessing.py`에서 payment를 주문 단위로 먼저 집계합니다.

In [ ]:
merged = (
    items
    .merge(orders, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(reviews, on="order_id", how="left")
    .merge(payments, on="order_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left", suffixes=("", "_seller"))
)

merged = merged.dropna(subset=["review_score", "product_category_name"])
print(merged.shape)
print(merged.isnull().sum().sort_values(ascending=False).head(10))

## 4. Delivery-derived variable

원본 notebook에서는 실제 고객 수령일과 예상 배송일의 차이를 `deliverdiff`로 계산했습니다.

In [ ]:
from datetime import timedelta

merged["order_estimated_delivery_date"] = (
    pd.to_datetime(merged["order_estimated_delivery_date"]) + timedelta(days=1)
)
merged["order_delivered_customer_date"] = pd.to_datetime(
    merged["order_delivered_customer_date"]
)

merged["deliverdiff"] = (
    merged["order_delivered_customer_date"]
    - merged["order_estimated_delivery_date"]
).dt.days

merged.loc[merged["order_status"] != "delivered", "order_status"] = "not delivered"

## 5. Low-review subset and category/state exploration

복구된 notebook의 EDA 흐름 중 대표적인 부분만 남겼습니다.

In [ ]:
low_review = merged.loc[
    (merged["review_score"] < 3.0)
    & (merged["order_status"] == "delivered")
].copy()

category_counts = low_review["product_category_name"].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=category_counts.index, y=category_counts.values)
plt.title("Low-review Orders by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Order rows")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
top_category_by_state = (
    merged.groupby(["seller_state", "product_category_name"])
    .size()
    .reset_index(name="count")
    .sort_values(["seller_state", "count"], ascending=[True, False])
)

top_category_per_state = (
    top_category_by_state
    .groupby("seller_state")
    .first()
    .reset_index()
)

delivery_score_corr = merged[["review_score", "deliverdiff"]].corr().iloc[0, 1]

late_delivery_by_state = (
    merged.groupby("customer_state")["deliverdiff"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="avg_delay_days")
)

low_score_by_state = (
    merged.groupby("customer_state")["review_score"]
    .mean()
    .sort_values()
    .head(10)
    .reset_index(name="avg_review_score")
)

delivery_score_corr, late_delivery_by_state.head(), low_score_by_state.head()

## 6. Outlier checks

원본 notebook에서 실제로 확인했던 배송/리뷰 이상 패턴 일부입니다.

In [ ]:
early_but_low_review = merged[
    (merged["deliverdiff"] < 0)
    & (merged["review_score"] < 3)
]

delay_threshold = (
    merged["deliverdiff"].mean()
    + 1.5 * merged["deliverdiff"].std()
)

very_late_but_five_star = merged[
    (merged["deliverdiff"] > delay_threshold)
    & (merged["review_score"] == 5)
]

fast_but_low_review = merged[
    (merged["deliverdiff"] <= 1)
    & (merged["review_score"] <= 2)
]

{
    "early_but_low_review": len(early_but_low_review),
    "very_late_but_five_star": len(very_late_but_five_star),
    "fast_but_low_review": len(fast_but_low_review),
}

## Notes

이 notebook은 당시 탐색 과정의 대표 코드만 정리한 **cleaned review notebook**입니다.  
프로젝트의 Seller 전략, HHI, 고객 유지, 배송 관련 최종 비즈니스 해석은 README와 `docs/` 문서를 함께 확인하세요.